# Chapitre 4 — Introduction à l'algorithmique
*NSI Première — cours à lire avant les exercices*

Ce cours reprend le diaporama du chapitre 4, dans le même ordre, mais en expliquant chaque idée pas à pas. Il ne suppose rien d'autre que savoir lire un petit programme Python avec `if`, `for`, `while` et des listes.

Quand on écrit un programme, on peut se poser quatre questions. Chacune est une partie du cours :

| Question | Nom de la notion | Partie |
|:---|:---|:---|
| Que doit faire exactement ce programme ? | la **spécification** | 2 |
| Est-ce qu'il s'arrête toujours ? | la **terminaison** | 3 |
| Est-ce qu'il donne toujours le bon résultat ? | la **correction** | 4 |
| Est-ce qu'il est rapide ? | la **complexité** | 5 |

La partie 6 applique les quatre questions à un algorithme célèbre : la **recherche par dichotomie**. La partie 7 est la fiche à retenir.

## 1. Algorithme et programme

### Le besoin
Avant d'écrire du code, il faut savoir **quelle méthode** on va suivre. Par exemple, pour trouver le plus grand nombre d'une liste, on peut le faire à la main sans ordinateur : on retient le premier nombre, puis on regarde les autres un par un, et dès qu'on en voit un plus grand, on le retient à sa place. Cette méthode existe avant tout langage de programmation.

### Définition
Un **algorithme** est une suite **finie** d'instructions qui permet de résoudre un problème. Il transforme des **entrées** (les données qu'on lui donne) en une **sortie** (le résultat).

```
Entrées (données)  →  Algorithme (traitement)  →  Sortie (résultat)
```

Un **programme** est l'écriture d'un algorithme dans un langage précis, par exemple Python. On dit que le programme est une **implémentation** de l'algorithme. Le même algorithme peut être écrit en Python, en Java ou en **pseudo-code** (du français structuré, comme dans le diaporama).

### Exemple : le maximum d'une liste

En pseudo-code :
```
fonction maximum(L)
    max ← L[0]
    pour chaque x dans L faire
        si x > max alors
            max ← x
    renvoyer max
```
La flèche `←` veut dire « reçoit » : c'est l'affectation, le `=` de Python.

In [1]:
def maximum(t):
    m = t[0]            # on retient le premier élément
    for x in t:         # on regarde les éléments un par un
        if x > m:       # s'il est plus grand que celui qu'on retient,
            m = x       # on le retient à sa place
    return m

print(maximum([3, 8, 2]))
print(maximum([-5, -2, -9]))

8
-2


Remarque : la boucle regarde aussi `t[0]`, qu'on a déjà retenu. Ce n'est pas grave : `t[0] > t[0]` est faux, donc rien ne change. (Le diaporama écrit `for x in t[1:]` pour sauter le premier élément ; le résultat est le même.)

## 2. La spécification

### Le besoin
Imagine qu'un camarade te donne la fonction `maximum` et que tu l'appelles avec une **liste vide** : `t[0]` n'existe pas, le programme plante. Est-ce que la fonction est fausse ? Non : elle n'a jamais été prévue pour une liste vide. Il faut donc **écrire noir sur blanc** ce que la fonction attend et ce qu'elle promet. C'est un contrat entre celui qui écrit la fonction et celui qui l'utilise.

### Définition
La **spécification** décrit **ce que doit faire** l'algorithme, **sans dire comment** il le fait. Elle a deux parties :
- la **précondition** : ce qu'on **suppose vrai sur les entrées**. C'est la responsabilité de celui qui appelle la fonction.
- la **postcondition** : ce qui **doit être vrai sur la sortie**, à condition que la précondition soit respectée. C'est la promesse de la fonction.

Pour `maximum(L)` :
- précondition : « L est une liste **non vide** d'entiers » ;
- postcondition : « le résultat est le plus grand élément de L », c'est-à-dire : il **appartient** à L, et il est **supérieur ou égal à tous** les éléments de L.

Si on appelle la fonction **sans respecter la précondition** (avec une liste vide), la fonction ne promet plus rien : ce n'est pas sa faute.

### En Python
- On écrit la spécification dans la **docstring** : le texte entre `"""` juste sous le `def`.
- On peut **vérifier** la précondition pendant l'exécution avec `assert condition, "message"` : si la condition est fausse, le programme s'arrête avec une erreur `AssertionError` et le message.

In [2]:
def maximum(l):
    """Renvoie le plus grand élément de la liste l.
    Précondition : l est une liste non vide de nombres.
    Postcondition : le résultat appartient à l et est >= à tous ses éléments.
    """
    assert len(l) > 0, "précondition non respectée : la liste est vide"
    m = l[0]
    for x in l:
        if x > m:
            m = x
    return m

print(maximum([4, 1, 7]))
try:
    maximum([])                 # on ne respecte pas la précondition
except AssertionError as erreur:
    print("AssertionError :", erreur)

7
AssertionError : précondition non respectée : la liste est vide


⚠️ Dans le diaporama, la ligne `assert len(t) > 0, #précondition non respectée` contient deux erreurs de frappe : la liste s'appelle `l` et non `t`, et après la virgule il faut un **message entre guillemets**, pas un commentaire. Ci-dessus, la version corrigée. (Le `try ... except` sert seulement à afficher l'erreur sans arrêter le notebook.)

## 3. La terminaison

### Le besoin
Un programme qui ne s'arrête jamais (une **boucle infinie**) ne donne jamais de résultat : l'ordinateur tourne sans fin. Exemple :
```python
n = 7
while n != 0:
    n = n - 2      # n vaut 5, 3, 1, -1, -3, ... et ne vaut jamais 0
```
On veut pouvoir **prouver**, en lisant le code, qu'une boucle s'arrête, sans avoir à l'essayer avec toutes les valeurs possibles (il y en a une infinité).

### Définition
Un algorithme **termine** si, **pour toute entrée qui respecte la précondition**, il s'arrête après un nombre fini d'étapes.

- Une boucle **`for`** termine toujours : on sait à l'avance combien de tours elle fait (un par élément de la liste, ou un par valeur du `range`).
- Le danger vient des boucles **`while`** : elles tournent tant que leur condition est vraie, et rien ne garantit qu'elle devienne fausse un jour.

### Le variant de boucle

**L'idée.** Pense à un compte à rebours : 10, 9, 8… Un nombre **entier** qui **diminue à chaque tour** ne peut pas diminuer indéfiniment sans finir par passer sous 0. Si on sait que la boucle s'arrête dès que ce nombre passe sous 0, alors la boucle s'arrête forcément.

**Définition.** Un **variant** d'une boucle `while` est une quantité calculée avec les variables du programme, qui vérifie **trois conditions** :
1. c'est un nombre **entier** ;
2. il **diminue strictement** à chaque tour de boucle (il perd au moins 1) ;
3. s'il devient **inférieur ou égal à 0**, la condition du `while` est fausse, donc la boucle s'arrête.

**Si on trouve un variant, on a prouvé que la boucle termine.**

### Méthode pour trouver un variant
1. Regarde la **condition** du `while`. Elle a souvent la forme `A > B` (ou `A < B`).
2. Essaie la **différence** qui doit rester positive pour que la boucle continue : `A - B` (ou `B - A`).
3. Vérifie les trois conditions : est-ce un entier ? diminue-t-il à **chaque** tour ? la boucle s'arrête-t-elle quand il atteint 0 ?

### Exemples du diaporama

| Boucle | Variant | Justification |
|:---|:---|:---|
| `while n > 0:` `n = n - 3` | `n` | `n` est entier, perd 3 à chaque tour ; si `n <= 0`, la condition `n > 0` est fausse |
| `while n > 10:` `n = n - 2` | `n - 10` | perd 2 à chaque tour ; si `n - 10 <= 0`, alors `n <= 10` et la boucle s'arrête |
| `while n > b:` `n = n - b` (avec `b` entier strictement positif) | `n - b` | perd `b` (au moins 1) à chaque tour ; si `n - b <= 0`, alors `n <= b` et la boucle s'arrête |
| `while n < 10:` `n = n - 2` | **aucun** | `n` diminue, donc reste toujours `< 10` : la boucle est **infinie** dès qu'on y entre |
| `while n != 0:` `n = n - 3` | **aucun** | si `n` vaut 7 au départ : 4, 1, -2, -5… `n` saute 0 et ne s'arrête jamais. Elle ne termine que si `n` est un multiple positif de 3 |
| `while n > a and p > b:` à chaque tour, soit `n` soit `p` diminue de 1 (selon que `n` est pair ou non) | `(n - a) + (p - b)` | à chaque tour, **l'un des deux** diminue de 1, donc la somme perd 1 ; si elle est `<= 0`, l'une des deux parenthèses est `<= 0`, donc `n <= a` ou `p <= b`, et la condition `and` est fausse |

**Retenir.** La condition `n != 0` est dangereuse : elle ne s'arrête que si `n` tombe **exactement** sur 0. La remplacer par `n > 0` règle le problème : `n` devient alors un variant.

### Un cas où personne ne sait : Syracuse

In [3]:
def syracuse(n):
    duree_de_vol = 0
    while n != 1:
        if n % 2 == 0:          # n pair : on le divise par 2
            n = n // 2
        else:                   # n impair : on le multiplie par 3 et on ajoute 1
            n = 3 * n + 1
        duree_de_vol = duree_de_vol + 1
    return duree_de_vol

print(syracuse(6))     # 6, 3, 10, 5, 16, 8, 4, 2, 1 : 8 tours
print(syracuse(27))

8
111


`n` ne diminue pas à chaque tour (il remonte quand il est impair), donc `n` n'est pas un variant. **Personne n'a jamais trouvé de variant** pour cette boucle, mais personne n'a prouvé non plus qu'il n'en existe pas : on ne sait pas si elle s'arrête pour **toutes** les valeurs de `n`. On l'a vérifié par ordinateur pour des milliards de valeurs, mais un essai n'est pas une preuve. C'est un problème de mathématiques encore ouvert.

## 4. La correction

### Le besoin
Qu'un programme s'arrête ne suffit pas : il faut aussi que le résultat soit **juste**. Et **tester ne suffit pas** pour en être sûr. Regarde cette version de `maximum` qui commence avec `m = 0` au lieu de `m = t[0]` :

In [4]:
def maximum_faux(t):
    m = 0
    for x in t:
        if x > m:
            m = x
    return m

print(maximum_faux([3, 8, 2]))        # 8 : juste
print(maximum_faux([1, 5, 4, 9]))     # 9 : juste
print(maximum_faux([-3, -1, -5]))     # 0 : FAUX, 0 n'est même pas dans la liste

8
9
0


Les deux premiers tests passent, et pourtant la fonction est fausse. Un test peut **montrer** qu'il y a une erreur, mais aucun nombre de tests ne peut **prouver** qu'il n'y en a pas (on ne peut pas essayer toutes les listes). Pour prouver qu'un programme est juste, on raisonne avec un **invariant de boucle**.

### Définition
Un algorithme est **correct** si, pour toute entrée qui respecte la précondition, il **termine** et renvoie un résultat qui **respecte la postcondition**.

Donc : **correction = terminaison + bon résultat.**

### L'invariant de boucle

**L'idée.** On cherche une phrase qui décrit **ce que les variables contiennent à chaque étape**, et qui reste **toujours vraie** d'un tour à l'autre, comme un résumé du travail déjà fait. Pour une somme, par exemple : « `s` contient la somme des éléments déjà parcourus ».

**Définition.** Un **invariant** est une propriété **vraie avant la boucle**, et qui **reste vraie après chaque tour**.

Pour prouver la correction avec un invariant, on montre **trois choses** :
1. **Initialisation** : l'invariant est vrai **avant le premier tour**.
2. **Conservation** : **si** il est vrai au début d'un tour, **alors** il est encore vrai à la fin de ce tour.
3. **Conclusion** : quand la boucle s'arrête, l'invariant, combiné avec la condition d'arrêt, **donne la postcondition**.

C'est comme une échelle : on sait monter sur le premier barreau (initialisation), et depuis n'importe quel barreau on sait monter sur le suivant (conservation) ; donc on arrive en haut (conclusion).

Pour une boucle `for`, il vaut mieux la réécrire d'abord en boucle `while` avec un compteur `i` : on voit mieux ce qui change à chaque tour.

### Exemple : la somme d'une liste

In [5]:
def somme(l):
    s = 0
    i = 0
    while i < len(l):
        s = s + l[i]
        i = i + 1
    return s

print(somme([3, 1, 4, 5]))

13


- **Précondition** : `l` est une liste d'entiers.
- **Postcondition** : `s = l[0] + l[1] + ... + l[len(l) - 1]` (la somme de **tous** les éléments).
- **Invariant** : `s = l[0] + ... + l[i - 1]`, autrement dit « `s` contient la somme des `i` premiers éléments ».

**Trace d'exécution** de `somme([3, 1, 4, 5])` : à la fin de chaque tour, on vérifie que `s` est bien la somme des `i` premiers éléments.

| Tour | i (début) | l[i] | s avant | s après | i (fin) | somme des i premiers |
|:---|:---|:---|:---|:---|:---|:---|
| 1 | 0 | 3 | 0 | 3 | 1 | 3 ✔ |
| 2 | 1 | 1 | 3 | 4 | 2 | 3 + 1 = 4 ✔ |
| 3 | 2 | 4 | 4 | 8 | 3 | 3 + 1 + 4 = 8 ✔ |
| 4 | 3 | 5 | 8 | 13 | 4 | 3 + 1 + 4 + 5 = 13 ✔ |

**Preuve** :
1. **Initialisation** : avant la boucle, `i = 0` : l'invariant parle de la somme des **0 premiers** éléments, c'est-à-dire d'aucun élément, qui vaut 0. Et `s = 0`. C'est vrai.
2. **Conservation** : on suppose qu'au début d'un tour, `s = l[0] + ... + l[i - 1]`. Pendant le tour, on ajoute `l[i]` à `s`, donc `s = l[0] + ... + l[i - 1] + l[i]` ; puis `i` augmente de 1. Avec le nouveau `i`, cela s'écrit encore `s = l[0] + ... + l[i - 1]`. L'invariant est toujours vrai.
3. **Conclusion** : la boucle s'arrête quand `i < len(l)` devient faux ; comme `i` augmente de 1 en partant de 0, on a alors `i = len(l)`. L'invariant dit donc `s = l[0] + ... + l[len(l) - 1]` : c'est la postcondition.

(Et la boucle termine : `len(l) - i` est un variant.)

### Méthode pour trouver un invariant
Demande-toi : **« après `i` tours, qu'est-ce que la variable importante contient ? »** La réponse, écrite avec `i`, est presque toujours l'invariant. Une trace d'exécution (un tableau comme ci-dessus) aide beaucoup à le voir.

## 5. La complexité

### Le besoin
Deux programmes peuvent donner le même résultat, mais l'un met une seconde et l'autre une heure. On veut pouvoir **comparer leur rapidité en lisant le code**, sans dépendre de l'ordinateur utilisé. Surtout, on veut savoir ce qui se passe **quand les données deviennent grandes** : un programme qui marche bien sur une liste de 10 éléments, que devient-il sur une liste de 100 000 éléments ?

### Définition
La **complexité en temps** d'un algorithme mesure le **nombre d'opérations élémentaires** qu'il effectue, en fonction de la **taille `n`** de l'entrée.
- La **taille `n`** est ce qu'on donne au programme : le plus souvent la **longueur de la liste** ou de la **chaîne de caractères**.
- Une **opération élémentaire** est une opération qui prend un temps fixe, quelle que soit la taille de l'entrée :
  1. une affectation (`x = ...`) ;
  2. la modification d'une case (`l[i] = ...`) ;
  3. des méthodes comme `l.pop()`, `l.append(x)` ;
  4. un calcul comme `a + b`, un test comme `a == b`.

La complexité répond donc à la question : **« si je donne une entrée de taille `n`, combien d'opérations le programme va-t-il faire ? »** On exprime le nombre d'opérations **en fonction de `n`** (et pas l'inverse).

### D'où viennent les opérations ?
Regardons la fonction de la diapo 21, qui cherche `x` dans une liste `l` :
```python
def indice(l, x):
    for i in range(len(l)):   # à chaque tour : on passe au i suivant
        if l[i] == x:         # à chaque tour : 1 test
            return True       # une seule fois
    return False              # une seule fois
```
Il y a deux sortes d'opérations :
1. **Les opérations dans la boucle** sont faites **à chaque tour**. Ici, il y en a environ 2 par tour : passer au `i` suivant, puis faire le test `l[i] == x`.
2. **Les opérations hors de la boucle** sont faites **une seule fois**, quelle que soit la taille de la liste. Ici, c'est le `return` final. Dans d'autres programmes, ce sera par exemple une affectation de départ comme `compteur = 0`.

Dans le pire des cas, `x` n'est pas dans la liste, et la boucle fait `n` tours. Le nombre d'opérations vaut donc **2 × n + 1** :

| Taille `n` | Dans la boucle (2 × n) | Hors de la boucle | Total |
|:---|:---|:---|:---|
| 10 | 20 | 1 | 21 |
| 1 000 | 2 000 | 1 | 2 001 |
| 100 000 | 200 000 | 1 | 200 001 |

### Ce qu'on ignore
Dans ce total, deux choses **ne dépendent pas de `n`**. On les ignore :
- **Les opérations hors de la boucle** (le « + 1 »). Il y en a toujours le même nombre. Quand `n` devient grand, elles ne pèsent presque rien : 1 opération sur 200 001.
- **Le nombre d'opérations dans un tour** (le « × 2 »). Chaque tour fait toujours le même petit nombre d'opérations. Si `n` est multiplié par 10, alors `2 × n` est multiplié par 10 aussi, exactement comme `n`. Ce « × 2 » ne change donc pas **la façon dont le nombre d'opérations grandit**.

(C'est pour la même raison que la vitesse de l'ordinateur ne compte pas : un ordinateur 2 fois plus rapide divise tous les temps par 2, mais ne change pas la façon dont ils grandissent.)

Ce qui reste, c'est **le nombre de tours de boucle** : c'est la seule chose qui grandit avec `n`. C'est pour cela qu'en pratique on compte les **itérations** (les tours de boucle), et pas chaque opération une par une.

### Le pire des cas

In [6]:
def indice(l, x):
    for i in range(len(l)):
        if l[i] == x:
            return True
    return False

print(indice([7, 2, 9, 4], 7))   # trouvé au 1er tour : on s'arrête tout de suite
print(indice([7, 2, 9, 4], 5))   # pas trouvé : on fait les 4 tours

True
False


Le nombre de tours ne dépend pas seulement de la taille, il dépend aussi des valeurs : si `x` est au début, un seul tour suffit. On s'intéresse toujours au **pire des cas**, celui qui fait le plus de tours. C'est une garantie : « ça ne sera jamais pire que ça ».

### Situation 1 : chercher un élément en parcourant la liste
C'est la fonction `indice` ci-dessus. Dans le pire des cas, `x` n'est pas dans la liste et il faut **regarder chaque case** :
- liste de 8 éléments : 8 tours ;
- liste de 1 000 éléments : 1 000 tours ;
- liste de 100 000 éléments : 100 000 tours.

👉 **Le nombre de tours est égal à `n`.** C'est un algorithme **linéaire**.

### Situation 2 : chercher un élément dans une liste triée, par dichotomie
On verra cet algorithme en détail dans la partie 6. Il suffit ici de savoir qu'à chaque tour, il **jette la moitié de la zone de recherche**. Avec 8 éléments :
8 cases → 4 cases → 2 cases → 1 case, soit **3 coupes**, plus un dernier tour pour regarder la case restante.
- liste de 8 éléments : environ 3 tours ;
- liste de 1 000 éléments : environ 10 tours (1000 → 500 → 250 → 125 → 62 → 31 → 15 → 7 → 3 → 1) ;
- liste de 100 000 éléments : environ 17 tours.

Le nombre de tours est **le nombre de fois qu'on peut couper `n` en deux avant d'arriver à 1**. Pour mieux le voir, on peut se poser la question à l'envers : avec `k` tours, quelle taille de liste peut-on traiter ?
- 1 tour → 2 cases, soit 2¹ ;
- 2 tours → 4 cases, soit 2² ;
- 3 tours → 8 cases, soit 2³ ;
- `k` tours → `n = 2ᵏ` cases.

Si `n = 2ᵏ`, alors `k` est le nombre qu'on appelle en maths le **logarithme en base 2** de `n`, noté **`log₂ n`**. Le logarithme est l'opération qui « défait » la puissance de 2.

**Définition :** `log₂(n) = k` veut dire `2ᵏ = n`. Autrement dit, `log₂(n)` est la puissance à laquelle il faut élever 2 pour obtenir `n`.
- `log₂(2) = 1`, car 2¹ = 2 ;
- `log₂(4) = 2`, car 2² = 4 ;
- `log₂(8) = 3`, car 2³ = 8 ;
- `log₂(1024) = 10`, car 2¹⁰ = 1024.

👉 **Le nombre de tours est environ `log₂ n`.** C'est un algorithme **logarithmique**.

### Situation 3 : deux boucles imbriquées
On veut savoir si une liste contient un doublon. On compare chaque élément à tous les autres :

In [ ]:
def a_un_doublon(L):
    for i in range(len(L)):
        for j in range(len(L)):
            if i != j and L[i] == L[j]:
                return True
    return False

print(a_un_doublon([3, 1, 4, 1]))   # le 1 apparaît deux fois : True
print(a_un_doublon([3, 1, 4]))      # pas de doublon : on fait les 3 × 3 tours, False

True
False


Pour **chacun** des `n` tours de la boucle extérieure, la boucle intérieure refait `n` tours. Dans le pire des cas, il n'y a pas de doublon et on fait tout :
- liste de 8 éléments : 8 × 8 = 64 tours ;
- liste de 1 000 éléments : 1 000 × 1 000 = 1 000 000 tours ;
- liste de 100 000 éléments : 10 milliards de tours.

👉 **Le nombre de tours est `n × n = n²`.** C'est un algorithme **quadratique**. Parcourir une matrice `n × n` case par case donne le même résultat.

### Situation 4 : énumérer toutes les possibilités
On veut écrire toutes les suites de `n` chiffres 0 ou 1. Chaque chiffre a 2 choix possibles :
- `n = 1` : 0, 1 → 2 suites ;
- `n = 2` : 00, 01, 10, 11 → 4 suites ;
- `n = 3` : 000, 001, 010, 011, 100, 101, 110, 111 → 8 suites ;
- `n = 10` : 1 024 suites ;
- `n = 100` : environ 10³⁰ suites (un 1 suivi de 30 zéros).

À chaque chiffre ajouté, le nombre de suites **double**. Il vaut donc 2 × 2 × … × 2, `n` fois, soit `2ⁿ`. C'est ce qui arrive quand on veut essayer toutes les façons de choisir des objets (le problème du sac à dos) : chaque objet est pris (1) ou laissé (0).

👉 **Le nombre de possibilités est `2ⁿ`.** C'est un algorithme **exponentiel**.

### On ne garde que l'ordre de grandeur
Ce qu'on a ignoré plus haut s'écrit avec la notation **`O(...)`**, qu'on lit « grand O de … ». `O(n)` veut dire « de l'ordre de `n` » : on ne garde que la façon dont le nombre d'opérations grandit avec `n`.

Deux règles de simplification :
- **On ignore les nombres qui multiplient.** C'est le « × 2 » de tout à l'heure : `2n` opérations ou `n` opérations, c'est le même ordre, `O(n)`.
- **On ne garde que le terme qui grandit le plus vite.** C'est le « + 1 » de tout à l'heure : `2n + 1` devient `O(n)`. De même, `n² + 2n + 1` devient `O(n²)`. Pour `n = 1000`, `n²` vaut un million et `2n + 1` vaut 2001 : ce second terme ne pèse presque rien.

### Les ordres de grandeur à connaître
Ce tableau regroupe les quatre situations, plus le cas sans boucle :

| Nom | Notation | Exemple |
|:---|:---|:---|
| constante | `O(1)` | quelques opérations, sans boucle (lire `t[0]`) |
| logarithmique | `O(log n)` | la dichotomie (situation 2) |
| linéaire | `O(n)` | **une** boucle qui parcourt la liste (situation 1) |
| quadratique | `O(n²)` | **deux** boucles imbriquées (situation 3) ; parcourir une matrice `n × n` |
| exponentielle | `O(2ⁿ)` | énumérer toutes les suites de `n` chiffres 0 ou 1 (situation 4) |

### Pourquoi c'est important
On peut maintenant rassembler les comptages des quatre situations dans un seul tableau. Chaque **ligne** correspond à une taille de liste `n`. Chaque **colonne** correspond à une situation. Chaque **case** donne le nombre d'opérations, c'est-à-dire de tours :

| Taille `n` | Dichotomie `log₂ n` | Parcours `n` | Deux boucles `n²` | Toutes les possibilités `2ⁿ` |
|:---|:---|:---|:---|:---|
| 100 | 6,6 | 100 | 10 000 | beaucoup trop grand |
| 1 000 | 10 | 1 000 | 1 000 000 | beaucoup trop grand |
| 10 000 | 13,3 | 10 000 | 100 000 000 | beaucoup trop grand |
| 100 000 | 16,6 | 100 000 | 10 000 000 000 | beaucoup trop grand |

Deux remarques pour bien lire ce tableau :
- La colonne « Parcours » répète la colonne « Taille ». Ce n'est pas une erreur : un algorithme linéaire fait **autant de tours qu'il y a d'éléments**.
- 6,6 n'est pas un nombre entier de tours : c'est la valeur mathématique de `log₂ 100`. En pratique, cela donne 7 tours.

Pour le rendre concret, prenons un ordinateur qui fait environ un milliard d'opérations par seconde. Avec 100 000 éléments :
- la dichotomie (17 tours) et le parcours (100 000 tours) sont **instantanés** ;
- les deux boucles (10 milliards de tours) prennent **une dizaine de secondes** ;
- énumérer toutes les possibilités prendrait **plus que l'âge de l'univers**, même avec seulement `n = 100`.

👉 **Sur de grandes données, le choix de l'algorithme compte beaucoup plus que la vitesse de l'ordinateur.** Améliorer la complexité, c'est rendre le programme plus performant.

### Le piège : une boucle qui appelle une fonction
Compter les tours suffit seulement si **chaque tour coûte un nombre fixe d'opérations**. Supposons qu'à l'intérieur d'une boucle de `n` tours, on appelle une fonction qui contient elle-même une boucle sur la liste. Alors chaque tour ne coûte plus 2 opérations, il en coûte `n` :
```python
for v in L:                 # n tours
    if indice(L, v + 1):    # indice fait jusqu'à n tours à chaque appel
        ...
```
Au total, cela fait `n` tours × `n` opérations = **`O(n²)`**, même s'il n'y a qu'**une seule** boucle visible.

### Méthode pour trouver la complexité
1. **Pas de boucle**, ou une boucle dont le nombre de tours ne dépend pas de `n` → `O(1)`.
2. **Une boucle** qui fait `n` tours, avec un travail fixe à chaque tour → `O(n)`.
3. **Deux boucles imbriquées** de chacune environ `n` tours → `n × n` → `O(n²)`.
4. **Une boucle qui divise la zone par 2 à chaque tour** → `O(log n)`.
5. **Une boucle qui appelle une fonction** : on **multiplie** le nombre de tours par la complexité de la fonction appelée.

**En une phrase :** la complexité, c'est l'ordre de grandeur du nombre d'opérations, exprimé en fonction de la taille `n` de l'entrée, dans le pire des cas.

## 6. Étude de cas : la recherche par dichotomie

### Le besoin
Chercher un élément dans une liste en la parcourant du début à la fin coûte `O(n)`. Mais si la liste est **triée**, on peut faire beaucoup mieux, comme quand on cherche un mot dans un dictionnaire papier : on ouvre au milieu, on regarde si le mot est avant ou après, et on recommence **dans la moitié qui reste**. On élimine la moitié des possibilités à chaque fois.

| Recherche dans une liste… | non triée | triée dans l'ordre croissant |
|:---|:---|:---|
| du premier élément | `O(1)` | `O(1)` |
| du maximum | `O(n)` (il faut tout regarder) | `O(1)` (c'est le dernier) |
| de l'indice d'un élément `x` | `O(n)` | `O(log n)` avec la dichotomie |

### L'algorithme
- **Précondition** : la liste `L` contient des entiers et est **triée dans l'ordre croissant**.
- **Postcondition** : le résultat est un indice `i` tel que `L[i] = x` ; si `x` n'est pas dans `L`, le résultat est `-1`.

On garde deux indices, `gauche` et `droite`, qui délimitent **la zone où `x` peut encore se trouver**. Au début, c'est toute la liste.

In [7]:
def dichotomie(L, x):
    gauche = 0
    droite = len(L) - 1
    while gauche <= droite:
        i = (gauche + droite) // 2      # l'indice du milieu de la zone
        if L[i] == x:
            return i                    # trouvé
        elif L[i] < x:
            gauche = i + 1              # x est plus grand : il ne peut être qu'à droite de i
        else:
            droite = i - 1              # x est plus petit : il ne peut être qu'à gauche de i
    return -1                           # la zone est vide : x n'est pas dans la liste

L = [1, 3, 5, 8, 9, 11, 14, 25, 34, 39]
print(dichotomie(L, 8))
print(dichotomie(L, 10))

3
-1


### Trace : recherche de `x = 8` dans `L` (l'exemple du diaporama)

```
indice :  0  1  2  3  4   5   6   7   8   9
L      :  1  3  5  8  9  11  14  25  34  39
```

| Tour | gauche | droite | i = (gauche + droite) // 2 | L[i] | Décision |
|:---|:---|:---|:---|:---|:---|
| 1 | 0 | 9 | 4 | 9 | 9 > 8 : `x` est à gauche, `droite = 3` |
| 2 | 0 | 3 | 1 | 3 | 3 < 8 : `x` est à droite, `gauche = 2` |
| 3 | 2 | 3 | 2 | 5 | 5 < 8 : `x` est à droite, `gauche = 3` |
| 4 | 3 | 3 | 3 | 8 | trouvé : on renvoie 3 |

`//` est la division **entière** : `(0 + 9) // 2` vaut 4 (et pas 4,5).

### Terminaison
**Variant** : `droite - gauche`, la largeur de la zone moins 1.
1. C'est un **entier**.
2. À chaque tour où on ne s'arrête pas, la zone **rétrécit** : soit `gauche` devient `i + 1`, qui est plus grand que l'ancien `gauche` (car `i >= gauche`) ; soit `droite` devient `i - 1`, qui est plus petit que l'ancien `droite` (car `i <= droite`). Donc `droite - gauche` diminue d'au moins 1.
3. Quand il devient **strictement négatif**, on a `gauche > droite`, et la condition `gauche <= droite` est fausse : la boucle s'arrête.

(Ici, la boucle s'arrête quand le variant passe sous 0, et pas quand il vaut 0 : quand `gauche = droite`, il reste encore une case à regarder. On peut prendre `droite - gauche + 1`, le nombre de cases de la zone, pour coller exactement à la définition.)

### Correction
**Invariant** : « si `x` est présent dans `L`, alors il est entre les indices `gauche` et `droite` ».
1. **Initialisation** : au début, `gauche = 0` et `droite = len(L) - 1` : la zone est toute la liste. Si `x` est dans `L`, il y est forcément.
2. **Conservation** : on suppose que `x`, s'il est présent, est entre `gauche` et `droite`.
   - Si `L[i] = x`, on renvoie `i` : c'est juste.
   - Si `L[i] < x` : comme la liste est **triée**, toutes les cases d'indice `<= i` contiennent des valeurs `<= L[i]`, donc `< x`. `x` ne peut pas y être ; il est donc entre `i + 1` et `droite`. On fait `gauche = i + 1` : l'invariant reste vrai.
   - Si `L[i] > x` : de même, `x` ne peut être qu'entre `gauche` et `i - 1`. On fait `droite = i - 1` : l'invariant reste vrai.
3. **Conclusion** : si la boucle s'arrête sans avoir trouvé `x`, c'est que `gauche > droite` : la zone est **vide**. D'après l'invariant, `x` n'est donc pas dans `L`, et renvoyer `-1` est juste.

C'est ici qu'on voit pourquoi la précondition « liste triée » est indispensable : sans elle, la conservation ne marche plus.

### Complexité : pourquoi `O(log n)` ?
À chaque tour, la zone est **coupée en deux** (au moins). Avec une liste de 1 000 éléments, la zone fait environ 1 000, puis 500, 250, 125, 62, 31, 15, 7, 3, 1 : au bout d'**environ 10 tours**, il ne reste qu'une case.

Le nombre de fois qu'on peut diviser `n` par 2 avant d'arriver à 1 s'appelle le **logarithme en base 2** de `n`, noté `log₂ n`. Par exemple `log₂ 1000 ≈ 10` et `log₂ 1 000 000 ≈ 20`. Dans le pire des cas, la dichotomie fait donc environ `log₂ n` tours : sa complexité est `O(log n)`.

Doubler la taille de la liste n'ajoute qu'**un seul** tour.

In [8]:
# On compte les tours de la dichotomie dans le pire des cas (x absent, plus grand que tout)
def nombre_de_tours(n):
    L = []
    for k in range(n):
        L.append(k)
    gauche = 0
    droite = n - 1
    tours = 0
    x = n                       # plus grand que tous les éléments
    while gauche <= droite:
        tours = tours + 1
        i = (gauche + droite) // 2
        if L[i] == x:
            return tours
        elif L[i] < x:
            gauche = i + 1
        else:
            droite = i - 1
    return tours

for n in [10, 100, 1000, 1000000]:
    print(n, "éléments :", nombre_de_tours(n), "tours")

10 éléments : 4 tours
100 éléments : 7 tours
1000 éléments : 10 tours
1000000 éléments : 20 tours


## 7. À retenir

| Notion | Ce que c'est | Comment on le montre |
|:---|:---|:---|
| **Spécification** | ce que fait l'algorithme : **précondition** (sur les entrées) et **postcondition** (sur la sortie) | docstring ; `assert` pour vérifier la précondition |
| **Terminaison** | l'algorithme s'arrête pour toute entrée qui respecte la précondition | un **variant** : un entier qui diminue strictement à chaque tour et qui arrête la boucle quand il atteint 0 |
| **Correction** | terminaison + le résultat respecte la postcondition | un **invariant** : initialisation, conservation, conclusion |
| **Complexité** | l'ordre de grandeur du nombre d'opérations selon la taille `n`, dans le pire des cas | une boucle → `O(n)` ; deux boucles imbriquées → `O(n²)` ; on divise par 2 à chaque tour → `O(log n)` |

**Réflexes pour les exercices :**
- **Variant** : regarde la condition du `while` (`A > B` → essaie `A - B`), puis vérifie : entier ? diminue à chaque tour ? arrête la boucle ?
- **Boucle qui ne termine pas** : trouve **une** valeur de départ pour laquelle elle tourne à l'infini (un contre-exemple).
- **Invariant** : « après `i` tours, que contient la variable ? » ; fais une trace pour le voir.
- **Fonction fausse** : cherche une entrée **qui respecte la précondition** mais qui donne un mauvais résultat.
- **Complexité** : compte les boucles imbriquées, et multiplie quand une boucle appelle une fonction qui contient elle-même une boucle.